# 09 - 网络层深度分解 (Network Layer Deep Dive)

**目的**: 将 HTTP 请求拆解为网络层每一个独立事件，消除网络层盲区

**事件链（本 Notebook 覆盖）**:
```
发起请求
  ├─ [DNS]     域名解析（已缓存通常 < 1ms，冷启动可达 10-50ms）
  ├─ [TCP]     连接建立（3-way handshake）
  ├─ [TLS]     握手（HTTPS 额外开销）
  ├─ [SEND]    请求体上传（payload size / 带宽）
  ├─ [TTFB]    首字节时间（服务端真实处理延迟）
  ├─ [STREAM]  SSE 首个 chunk 到达
  └─ [TTLB]    最后字节（完整响应接收）
```

**测量工具**: httpx event hooks + curl subprocess（交叉验证）

## 0. 配置参数

In [ ]:
import os
from datetime import datetime

# =============================================
# 目标端点（完整请求链）
# =============================================
ENDPOINTS = {
    "ai_engine_health":   {"url": "http://localhost:8002",  "path": "/health",       "method": "GET"},
    "api_gateway_health": {"url": "http://localhost:8001",  "path": "/health",       "method": "GET"},
    "nginx_health":       {"url": "http://localhost:80",    "path": "/health",       "method": "GET"},
    "qdrant_health":      {"url": "http://localhost:6333",  "path": "/healthz",      "method": "GET"},
    "apipro":             {"url": "https://api.apipro.ai",  "path": "/v1/models",   "method": "GET"},
}

# SSE 流式端点（测量 SSE 首 chunk 时间）
SSE_URL = "http://localhost:8002/api/v1/requirement/stream"
SSE_TEST_MESSAGE = "我需要采购一批服务器"
APIPRO_API_KEY = os.getenv("APIPRO_API_KEY", "")

# 测试轮数
N_ITERATIONS = 20

DATA_DIR = "../data"
os.makedirs(DATA_DIR, exist_ok=True)
RESULTS_FILE = f"{DATA_DIR}/network_layer_{datetime.now().strftime('%Y%m%d_%H%M')}.csv"

print("配置加载完成")

## 1. 依赖导入

In [ ]:
import httpx
import time
import subprocess
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
import re

matplotlib.rcParams['font.sans-serif'] = ['Arial Unicode MS', 'SimHei', 'DejaVu Sans']
matplotlib.rcParams['axes.unicode_minus'] = False

print("依赖导入完成")

## 2. httpx Event Hooks — 逐事件计时

httpx 提供 `event_hooks` 机制，可以在连接建立、请求发送、响应接收等每个事件上注入回调。

In [ ]:
class NetworkTimer:
    """利用 httpx 事件钩子，精确记录网络层各阶段时间戳"""

    def __init__(self):
        self.reset()

    def reset(self):
        self.t_start          = None
        self.t_connect        = None   # TCP 连接建立完成
        self.t_request_sent   = None   # 请求体发送完成
        self.t_first_byte     = None   # 响应首字节（TTFB）
        self.t_end            = None   # 响应完全接收

    def on_request(self, request):
        self.t_start = time.perf_counter()

    def on_response(self, response):
        # 在 response headers 接收到时触发（这是真正的 TTFB）
        self.t_first_byte = time.perf_counter()

    def summary(self) -> dict:
        """返回各阶段耗时（ms）"""
        if not self.t_start:
            return {}
        t_end = self.t_end or time.perf_counter()
        result = {"total_ms": (t_end - self.t_start) * 1000}
        if self.t_first_byte:
            result["ttfb_ms"] = (self.t_first_byte - self.t_start) * 1000
            result["download_ms"] = (t_end - self.t_first_byte) * 1000
        return result


def measure_endpoint(url: str, path: str, method: str = "GET",
                     headers: dict = None, json_body: dict = None) -> dict:
    """使用 httpx event hooks 精确测量一次 HTTP 请求的各阶段"""
    timer = NetworkTimer()

    try:
        with httpx.Client(
            event_hooks={
                "request": [timer.on_request],
                "response": [timer.on_response],
            },
            timeout=10.0,
            follow_redirects=True,
        ) as client:
            if method == "GET":
                resp = client.get(f"{url}{path}", headers=headers or {})
            else:
                resp = client.post(f"{url}{path}", json=json_body, headers=headers or {})

            timer.t_end = time.perf_counter()
            timing = timer.summary()
            timing["http_status"] = resp.status_code
            timing["success"] = resp.status_code < 400
            timing["response_size_bytes"] = len(resp.content)
            return timing
    except Exception as e:
        timer.t_end = time.perf_counter()
        timing = timer.summary()
        timing["success"] = False
        timing["error"] = str(e)[:200]
        return timing


print(f"测量各端点网络延迟 ({N_ITERATIONS} 轮)")
print("=" * 70)

all_results = []

for ep_name, ep_cfg in ENDPOINTS.items():
    headers = {}
    if ep_name == "apipro" and APIPRO_API_KEY:
        headers = {"Authorization": f"Bearer {APIPRO_API_KEY}"}
    elif ep_name == "apipro" and not APIPRO_API_KEY:
        print(f"  跳过 {ep_name}: APIPRO_API_KEY 未配置")
        continue

    ep_times = []
    for i in range(N_ITERATIONS):
        r = measure_endpoint(ep_cfg["url"], ep_cfg["path"], ep_cfg["method"], headers)
        r["endpoint"] = ep_name
        r["iteration"] = i + 1
        all_results.append(r)
        if r.get("success"):
            ep_times.append(r["total_ms"])

    if ep_times:
        print(f"  {ep_name:25s}: TTFB P50={np.percentile([r.get('ttfb_ms', 0) for r in all_results[-N_ITERATIONS:] if r.get('ttfb_ms')], 50) if any(r.get('ttfb_ms') for r in all_results[-N_ITERATIONS:]) else 'N/A'}ms | 总计 P50={np.percentile(ep_times, 50):.1f}ms")
    else:
        print(f"  {ep_name:25s}: 不可达")

df_network = pd.DataFrame(all_results)
print("\n测量完成")

## 3. curl 交叉验证 — 获取 DNS/Connect/TTFB 精确值

curl 内置 `--write-out` 格式可以输出每个阶段的精确时间，是最可靠的网络层诊断工具。

In [ ]:
# curl 时间格式变量说明：
# time_namelookup  : DNS 解析耗时
# time_connect     : TCP 握手耗时（从0开始计，含DNS）
# time_appconnect  : TLS 握手耗时（仅 HTTPS，含以上所有）
# time_pretransfer : 开始传输请求体时（所有协商完成）
# time_starttransfer: 响应首字节时间（TTFB，含服务端处理）
# time_total       : 总耗时

CURL_FORMAT = (
    'dns_ms:%(time_namelookup)s,'
    'connect_ms:%(time_connect)s,'
    'tls_ms:%(time_appconnect)s,'
    'pretransfer_ms:%(time_pretransfer)s,'
    'ttfb_ms:%(time_starttransfer)s,'
    'total_ms:%(time_total)s,'
    'size_download:%(size_download)s'
)


def curl_measure(url: str, headers: dict = None, n: int = 5) -> list:
    """用 curl 精确测量 DNS/TCP/TLS/TTFB，返回 n 次测量结果"""
    results = []
    header_args = ""
    if headers:
        for k, v in headers.items():
            header_args += f' -H "{k}: {v}"'

    cmd = f'curl -s -o /dev/null -w "{CURL_FORMAT}" {header_args} "{url}"'

    for i in range(n):
        try:
            out = subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=10)
            if out.returncode == 0:
                # 解析输出
                r = {}
                for part in out.stdout.strip().split(","):
                    if ":" in part:
                        k, v = part.split(":", 1)
                        try:
                            # curl 输出秒，转换为毫秒
                            r[k.strip()] = float(v.strip()) * 1000
                        except ValueError:
                            r[k.strip()] = v.strip()
                # 计算各阶段独立耗时
                r["dns_only_ms"]     = r.get("dns_ms", 0)
                r["tcp_only_ms"]     = r.get("connect_ms", 0) - r.get("dns_ms", 0)
                r["tls_only_ms"]     = max(0, r.get("tls_ms", 0) - r.get("connect_ms", 0))
                r["server_proc_ms"]  = r.get("ttfb_ms", 0) - r.get("pretransfer_ms", 0)
                r["download_ms"]     = r.get("total_ms", 0) - r.get("ttfb_ms", 0)
                r["iteration"] = i + 1
                results.append(r)
        except Exception as e:
            results.append({"error": str(e), "iteration": i + 1})

    return results


print("curl 精确测量（5轮）")
print("=" * 70)

curl_targets = [
    ("AI Engine (localhost)",  "http://localhost:8002/health",  {}),
    ("API Gateway (localhost)","http://localhost:8001/health",  {}),
]
if APIPRO_API_KEY:
    curl_targets.append(("APIPro (外网)", "https://api.apipro.ai/v1/models", {"Authorization": f"Bearer {APIPRO_API_KEY}"}))

curl_all_results = []
for label, url, headers in curl_targets:
    results = curl_measure(url, headers, n=5)
    valid = [r for r in results if "error" not in r]

    for r in valid:
        r["endpoint"] = label
        curl_all_results.extend([r])

    if valid:
        print(f"\n[{label}]")
        print(f"  DNS 解析:   {np.mean([r['dns_only_ms'] for r in valid]):.2f} ms (均值)")
        print(f"  TCP 握手:   {np.mean([r['tcp_only_ms'] for r in valid]):.2f} ms")
        print(f"  TLS 握手:   {np.mean([r['tls_only_ms'] for r in valid]):.2f} ms")
        print(f"  服务端处理: {np.mean([r['server_proc_ms'] for r in valid]):.2f} ms")
        print(f"  响应下载:   {np.mean([r['download_ms'] for r in valid]):.2f} ms")
        print(f"  总计:       {np.mean([r['total_ms'] for r in valid]):.2f} ms")
    else:
        print(f"\n[{label}]: 不可达")

df_curl = pd.DataFrame(curl_all_results)

## 4. SSE 流式连接 — 首 Chunk 到达时间

In [ ]:
def measure_sse_first_chunk(url: str, message: str, n: int = 3) -> list:
    """测量 SSE 连接从发起到收到第一个内容 chunk 的时间"""
    results = []
    import uuid

    for i in range(n):
        payload = {"message": message, "session_id": str(uuid.uuid4()), "stream": True}
        t_start = time.perf_counter()
        t_connect = None
        t_first_chunk = None
        t_end = None
        chunks_received = 0
        error = None

        try:
            with httpx.Client(timeout=60.0) as client:
                with client.stream("POST", url, json=payload) as resp:
                    t_connect = time.perf_counter()  # SSE 连接建立（收到 response headers）

                    for line in resp.iter_lines():
                        if not line:
                            continue
                        if line.startswith("data:"):
                            data_str = line[5:].strip()
                            if data_str and data_str != "[DONE]":
                                try:
                                    chunk_data = json.loads(data_str)
                                    if chunk_data.get("type") == "content" or chunk_data.get("content"):
                                        if t_first_chunk is None:
                                            t_first_chunk = time.perf_counter()
                                        chunks_received += 1
                                except Exception:
                                    if t_first_chunk is None:
                                        t_first_chunk = time.perf_counter()
                                    chunks_received += 1
                        if chunks_received >= 3:  # 只测前几个 chunk，不等全部完成
                            break

        except Exception as e:
            error = str(e)[:200]

        t_end = time.perf_counter()

        result = {
            "iteration": i + 1,
            "total_ms": (t_end - t_start) * 1000,
            "sse_connect_ms": (t_connect - t_start) * 1000 if t_connect else None,
            "first_chunk_ms": (t_first_chunk - t_start) * 1000 if t_first_chunk else None,
            "chunks_received": chunks_received,
            "success": error is None and t_first_chunk is not None,
            "error": error,
        }
        results.append(result)
        status = "OK" if result["success"] else "FAIL"
        fc = f"{result['first_chunk_ms']:.0f}ms" if result["first_chunk_ms"] else "N/A"
        print(f"  [{status}] 轮{i+1}: SSE连接={result['sse_connect_ms']:.0f}ms | 首chunk={fc}")

    return results


print("SSE 流式连接测量")
print("=" * 60)
print(f"目标: {SSE_URL}")

sse_results = measure_sse_first_chunk(SSE_URL, SSE_TEST_MESSAGE, n=3)
df_sse = pd.DataFrame(sse_results)

valid_sse = df_sse[df_sse["success"]]
if not valid_sse.empty:
    print(f"\nSSE 统计:")
    print(f"  SSE 连接建立: {valid_sse['sse_connect_ms'].mean():.0f} ms")
    print(f"  首 chunk 到达: {valid_sse['first_chunk_ms'].mean():.0f} ms")
    print(f"  注: 首 chunk = TTFB + GreeterAgent + RequirementAgent 首次输出")

## 5. 网络层完整时序可视化

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle("网络层事件分解", fontsize=14)

# 图1: curl 各阶段堆叠柱状图（精确分解 DNS/TCP/TLS/服务端/下载）
ax1 = axes[0]
if not df_curl.empty and "endpoint" in df_curl.columns:
    endpoints = df_curl["endpoint"].unique()
    stages = ["dns_only_ms", "tcp_only_ms", "tls_only_ms", "server_proc_ms", "download_ms"]
    stage_labels = ["DNS", "TCP 握手", "TLS 握手", "服务端处理", "响应下载"]
    colors = ["#4CAF50", "#2196F3", "#9C27B0", "#F44336", "#FF9800"]

    x = range(len(endpoints))
    bottoms = [0] * len(endpoints)

    for stage, label, color in zip(stages, stage_labels, colors):
        vals = []
        for ep in endpoints:
            ep_data = df_curl[df_curl["endpoint"] == ep]
            if stage in ep_data.columns:
                vals.append(ep_data[stage].mean())
            else:
                vals.append(0)
        ax1.bar(x, vals, bottom=bottoms, label=label, color=color, alpha=0.85)
        for i, v in enumerate(vals):
            if v > 0.5:
                ax1.text(i, bottoms[i] + v/2, f"{v:.1f}", ha='center', va='center', fontsize=7, color='white')
        bottoms = [b + v for b, v in zip(bottoms, vals)]

    ax1.set_xticks(list(x))
    ax1.set_xticklabels(endpoints, rotation=15, ha='right', fontsize=8)
    ax1.set_ylabel("耗时 (ms)")
    ax1.set_title("各端点网络层分解 (curl 测量)")
    ax1.legend(loc="upper right", fontsize=8)
else:
    ax1.text(0.5, 0.5, "curl 数据不足", ha='center', va='center', transform=ax1.transAxes)
    ax1.set_title("各端点网络层分解")

# 图2: SSE 时序（连接 → 首chunk → 数据传输）
ax2 = axes[1]
if not valid_sse.empty:
    categories = ["SSE 连接建立", "首 chunk 到达", "总计(3chunk)"]
    p50_vals = [
        valid_sse["sse_connect_ms"].median(),
        valid_sse["first_chunk_ms"].median(),
        valid_sse["total_ms"].median(),
    ]
    colors2 = ["#4CAF50", "#FF9800", "#F44336"]
    bars = ax2.bar(range(3), p50_vals, color=colors2, alpha=0.85)
    ax2.set_xticks(range(3))
    ax2.set_xticklabels(categories)
    ax2.set_ylabel("耗时 (ms)")
    ax2.set_title("SSE 流式连接时序 (P50)")
    for bar, val in zip(bars, p50_vals):
        ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 10, f"{val:.0f}ms",
                 ha='center', fontsize=9)
else:
    ax2.text(0.5, 0.5, "SSE 端点不可达", ha='center', va='center', transform=ax2.transAxes)
    ax2.set_title("SSE 流式连接时序")

plt.tight_layout()
chart_file = f"../data/network_layer_chart_{datetime.now().strftime('%Y%m%d_%H%M')}.png"
plt.savefig(chart_file, dpi=150, bbox_inches="tight")
plt.show()
print(f"图表已保存: {chart_file}")

## 6. 保存结果

In [ ]:
all_data = []
if not df_network.empty:
    df_network["source"] = "httpx"
    all_data.append(df_network)
if not df_curl.empty:
    df_curl["source"] = "curl"
    all_data.append(df_curl)
if not df_sse.empty:
    df_sse["source"] = "sse"
    all_data.append(df_sse)

if all_data:
    pd.concat(all_data, ignore_index=True).to_csv(RESULTS_FILE, index=False)
    print(f"结果已保存: {RESULTS_FILE}")

print("\n网络层关键结论:")
print("  - 本地服务间通信 (localhost): 主要开销是 TCP 握手和服务端处理")
print("  - 外网 APIPro: 增加 DNS + TLS 开销（可通过连接池复用消除）")
print("  - SSE 首 chunk = 网络 + GreeterAgent + RequirementAgent 首次输出的累积")